In [ ]:
!pip install faiss-gpu

In [ ]:
import faiss
import numpy as np
import polars as pl
import torch
import time
import torch.nn as nn
import torch.nn.functional as F
import os
import time

## Stage 1 - Two-tower retrieval

### Architecture

In [ ]:
class UserTower(nn.Module):
    # num_dense_features = 25 (7 new features + 18 onehot features)
    def __init__(self, num_items, item_embed_dim=32, num_dense_features=25, final_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=item_embed_dim, padding_idx=0)
        self.gru = nn.GRU(input_size=item_embed_dim, hidden_size=64, batch_first=True)
        self.static_mlp = nn.Sequential(nn.Linear(num_dense_features, 32), nn.ReLU())
        self.fusion_layer = nn.Linear(64 + 32, final_dim)

    def forward(self, history_seq, static_features):
        seq_emb = self.item_embedding(history_seq)
        _, hidden = self.gru(seq_emb)
        user_history_vector = hidden.squeeze(0) 
        user_static_vector = self.static_mlp(static_features) 
        combined = torch.cat([user_history_vector, user_static_vector], dim=1)
        return self.fusion_layer(combined)


class ItemTower(nn.Module):
    def __init__(self, num_items, num_categories=2000, num_tags=500, item_embed_dim=32, nlp_embed_dim=512, final_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        
        # Categorical & Multi-Hot Tag Embeddings
        self.cat1_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.cat2_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.tag_embedding  = nn.Embedding(num_tags, 8, padding_idx=0)
        
        self.text_mlp = nn.Sequential(
            nn.Linear(nlp_embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )
        
        # Fusion Input: 
        # 32 (Item ID) + 8 (Cat1) + 8 (Cat2) + 8 (Pooled Tags) + 32 (Text MLP) + 1 (Log Duration) = 89
        self.fusion_layer = nn.Linear(32 + 8 + 8 + 8 + 32 + 1, final_dim)

    def forward(self, item_id, cat1_id, cat2_id, tags, duration_log, precomputed_text_vector):
        id_emb = self.item_embedding(item_id)
        c1_emb = self.cat1_embedding(cat1_id)
        c2_emb = self.cat2_embedding(cat2_id)
        
        # Multi-hot Bag-of-Tags Pooling:
        # tags shape: (Batch, MAX_TAGS) -> tag_emb: (Batch, MAX_TAGS, 8)
        tag_emb = self.tag_embedding(tags)
        # Sum across MAX_TAGS dimension -> shape: (Batch, 8)
        tag_emb = tag_emb.sum(dim=1) 
        
        text_feat = self.text_mlp(precomputed_text_vector)
        dur_feat = duration_log.unsqueeze(1) 
        
        combined = torch.cat([id_emb, c1_emb, c2_emb, tag_emb, text_feat, dur_feat], dim=1)
        return self.fusion_layer(combined)


class TwoTowerModel(nn.Module):
    def __init__(self, num_items, num_categories, num_tags, final_dim=64):
        super().__init__()
        self.user_tower = UserTower(num_items=num_items, final_dim=final_dim)
        self.item_tower = ItemTower(
            num_items=num_items, 
            num_categories=num_categories, 
            num_tags=num_tags, 
            final_dim=final_dim
        )

    def forward(self, history_seq, user_static, item_id, cat1_id, cat2_id, tags, duration_log, item_features):
        u_vector = self.user_tower(history_seq, user_static)
        i_vector = self.item_tower(item_id, cat1_id, cat2_id, tags, duration_log, item_features)
        
        u_vector = torch.nn.functional.normalize(u_vector, p=2, dim=1)  
        i_vector = torch.nn.functional.normalize(i_vector, p=2, dim=1)  
        
        similarity_matrix = torch.matmul(u_vector, i_vector.T)
        return similarity_matrix * 10.0

### Script helpers

In [ ]:
def build_faiss_index_gpu(df_item_embeddings, vector_dim=64):
    print("⚡ Building GPU FAISS Vector Index...")
    video_ids_map = df_item_embeddings["video_id"].to_numpy().astype(np.int64)
    item_vectors = np.array(df_item_embeddings["item_embedding_64d"].to_list(), dtype=np.float32)

    faiss.normalize_L2(item_vectors)

    cpu_index = faiss.IndexFlatIP(vector_dim)
    cpu_index.add(item_vectors)

    res = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

    print(f"✅ GPU FAISS Index Built! Total Indexed Items: {gpu_index.ntotal:,}")
    return gpu_index, video_ids_map


def retrieve_top_k_faiss(model, target_user_row, faiss_index, video_ids_map, device, k=200):
    model.eval()

    # 1. Compute dynamic User Vector (Shape: 1, 64)
    raw_history = target_user_row.get("history_sequence", [])
    if raw_history is None:
        raw_history = []
    MAX_SEQ_LEN = 100
    padded_history = ([0] * MAX_SEQ_LEN + raw_history)[-MAX_SEQ_LEN:]
    history_seq = torch.tensor([padded_history], dtype=torch.long).to(device)

    # --- FIX: Replicate the 25-feature string-to-float mapping logic ---
    activity_map = {"unknown": 0.0, "low_active": 1.0, "high_active": 2.0, "full_active": 3.0}
    follow_map = {'0': 0.0, '(0,10]': 1.0, '(10,50]': 2.0, '(50,100]': 3.0, '(100,150]': 4.0, '(150,250]': 5.0, '(250,500]': 6.0, '500+': 7.0}
    fans_map = {'0': 0.0, '[1,10)': 1.0, '[10,100)': 2.0, '[100,1k)': 3.0, '[1k,5k)': 4.0, '[5k,1w)': 5.0, '[1w,10w)': 6.0}
    friend_map = {'0': 0.0, '[1,5)': 1.0, '[5,30)': 2.0, '[30,60)': 3.0, '[60,120)': 4.0, '[120,250)': 5.0, '250+': 6.0}
    register_map = {'15-30': 1.0, '31-60': 2.0, '61-90': 3.0, '91-180': 4.0, '181-365': 5.0, '366-730': 6.0, '730+': 7.0}

    def safe_map(val, mapping_dict):
        return float(mapping_dict.get(str(val), 0.0))

    user_static_list = [
        safe_map(target_user_row.get("user_active_degree"), activity_map),
        float(target_user_row.get("is_live_streamer") or 0.0),
        float(target_user_row.get("is_video_author") or 0.0),
        safe_map(target_user_row.get("follow_user_num_range"), follow_map),
        safe_map(target_user_row.get("fans_user_num_range"), fans_map),
        safe_map(target_user_row.get("friend_user_num_range"), friend_map),
        safe_map(target_user_row.get("register_days_range"), register_map)
    ] + [float(target_user_row.get(f"onehot_feat{i}") or 0.0) for i in range(18)]

    # Cast to Tensor (Shape: 1, 25)
    user_static = torch.tensor([user_static_list], dtype=torch.float32).to(device)

    # Perform Forward Pass
    with torch.no_grad():
        user_vector = model.user_tower(history_seq, user_static)
        user_vector = torch.nn.functional.normalize(user_vector, p=2, dim=1)
        user_vector_np = user_vector.cpu().numpy().astype(np.float32)

    # 2. Query GPU FAISS Index
    scores, retrieved_indices = faiss_index.search(user_vector_np, k)

    # 3. Map array indices back to real video_ids
    top_k_items = video_ids_map[retrieved_indices[0]].tolist()

    return top_k_items

## Stage 2 - MTL Ranking

### Architecture

In [ ]:
class TargetAttention(nn.Module):
    """Deep Interest Network (DIN) Target Attention Module"""
    def __init__(self, embed_dim=32):
        super().__init__()
        self.attn_mlp = nn.Sequential(
            nn.Linear(embed_dim * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, target_item_emb, history_seq_embs):
        # target_item_emb: (Batch, Dim)
        # history_seq_embs: (Batch, SeqLen, Dim)
        seq_len = history_seq_embs.size(1)
        target_expanded = target_item_emb.unsqueeze(1).expand(-1, seq_len, -1)
        
        # Combine [Target, Sequence Item, Target - Sequence Item, Target * Sequence Item]
        concat_feat = torch.cat([
            target_expanded,
            history_seq_embs,
            target_expanded - history_seq_embs,
            target_expanded * history_seq_embs
        ], dim=-1)
        
        attn_weights = self.attn_mlp(concat_feat) # (Batch, SeqLen, 1)
        attn_weights = F.softmax(attn_weights, dim=1)
        
        # Weighted sum of past user history vectors
        user_interest_vector = torch.sum(attn_weights * history_seq_embs, dim=1)
        return user_interest_vector


class MMoELayer(nn.Module):
    """Multi-gate Mixture-of-Experts (MMoE) Layer"""
    def __init__(self, input_dim, num_experts=4, expert_hidden_dim=128, num_tasks=7):
        super().__init__()
        self.num_experts = num_experts
        self.num_tasks = num_tasks
        
        # 2. Two-Layer Experts: Added a second linear transformation and activation.
        # This increases the capacity of each expert to model nonlinear feature relationships.
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, expert_hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(expert_hidden_dim, expert_hidden_dim), # Second Layer
                nn.ReLU()
            ) for _ in range(num_experts)
        ])
        
        # Task Gating Networks
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, num_experts),
                nn.Softmax(dim=-1)
            ) for _ in range(num_tasks)
        ])

    def forward(self, x):
        # x shape: (Batch, input_dim)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1) # (Batch, NumExperts, HiddenDim)
        
        task_inputs = []
        for gate in self.gates:
            gate_weights = gate(x).unsqueeze(-1) # (Batch, NumExperts, 1)
            task_representation = torch.sum(gate_weights * expert_outputs, dim=1) # (Batch, HiddenDim)
            task_inputs.append(task_representation)
            
        return task_inputs


class MMoERankingModel(nn.Module):
    def __init__(self, num_items, num_categories, num_tags, item_embed_dim=32, nlp_embed_dim=512):
        super().__init__()
        
        # Embeddings
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        self.cat1_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.cat2_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.tag_embedding  = nn.Embedding(num_tags, 8, padding_idx=0)
        
        self.target_attention = TargetAttention(embed_dim=item_embed_dim)
        
        # NLP Reduction MLP
        self.text_mlp = nn.Sequential(nn.Linear(nlp_embed_dim, 32), nn.ReLU())
        
        # Original Input Dimension: 150
        raw_input_dim = 32 + 24 + 32 + 8 + 8 + 8 + 1 + 5 + 32
        
        # 1. Feature Preprocessing MLP:
        # We pass the raw concatenated features through this MLP before hitting the MMoE.
        # This lets distinct features interact globally before being routed to experts.
        mmoe_input_dim = 128
        self.feature_preprocessing = nn.Sequential(
            nn.Linear(raw_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, mmoe_input_dim),
            nn.ReLU()
        )
        
        # MMoE Engine (Now receives the 128-dim preprocessed features)
        self.mmoe = MMoELayer(input_dim=mmoe_input_dim, num_experts=4, expert_hidden_dim=64, num_tasks=7)
        
        # 3. Two-Layer Task Towers:
        # A helper function builds deeper task-specific heads instead of single linear layers.
        def build_task_tower(hidden_dim):
            return nn.Sequential(
                nn.Linear(hidden_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        # 7 Multi-Task Prediction Heads
        self.head_click = build_task_tower(64)       # Task 1: pCTR
        self.head_like = build_task_tower(64)        # Task 2: pLike
        self.head_comment = build_task_tower(64)     # Task 3: pComment
        self.head_forward = build_task_tower(64)     # Task 4: pForward
        self.head_hate = build_task_tower(64)        # Task 5: pHate
        self.head_long_view = build_task_tower(64)   # Task 6: pFinish / pLongView
        self.head_watch_time = build_task_tower(64)  # Task 7: Expected Watch Time

    def forward(self, history_seq, user_static, item_id, cat1_id, cat2_id, tags, item_stats, dur_log, nlp_vec):
        # Item Embeddings
        target_item_emb = self.item_embedding(item_id)
        history_seq_embs = self.item_embedding(history_seq)
        
        # Target Attention Over History
        user_interest = self.target_attention(target_item_emb, history_seq_embs)
        
        c1_emb = self.cat1_embedding(cat1_id)
        c2_emb = self.cat2_embedding(cat2_id)
        tag_emb = self.tag_embedding(tags).sum(dim=1)
        text_feat = self.text_mlp(nlp_vec)
        dur_feat = dur_log.unsqueeze(1)
        
        # Concatenate ALL User, Context, and Item features into a Single Representation
        dense_concat = torch.cat([
            user_interest, user_static, target_item_emb, 
            c1_emb, c2_emb, tag_emb, dur_feat, item_stats, text_feat
        ], dim=-1)
        
        # 1. Pass through Feature Preprocessing MLP
        preprocessed_features = self.feature_preprocessing(dense_concat)
        
        # MMoE Forward Pass
        task_representations = self.mmoe(preprocessed_features)
        
        # 4. Remove Sigmoids: BCEWithLogitsLoss requires raw logits!
        # 5. Remove F.relu on watch_time: We are predicting log-transformed watch time, 
        #    which can theoretically be modeled as a continuous real number.
        outputs = {
            "p_click_logits": self.head_click(task_representations[0]),
            "p_like_logits": self.head_like(task_representations[1]),
            "p_comment_logits": self.head_comment(task_representations[2]),
            "p_forward_logits": self.head_forward(task_representations[3]),
            "p_hate_logits": self.head_hate(task_representations[4]),
            "p_long_view_logits": self.head_long_view(task_representations[5]),
            "watch_time_log": self.head_watch_time(task_representations[6]) 
        }
        return outputs

class MultiTaskLoss(nn.Module):
    # Notice the watch_time weight is adjusted up slightly since log(ms) shrinks the loss scale.
    def __init__(self, weights={"click": 1.0, "like": 2.0, "comment": 2.0, "forward": 3.0, "hate": 3.0, "long_view": 1.5, "watch_time_log": 0.5}):
        super().__init__()
        # 4. Replace BCELoss with BCEWithLogitsLoss for far better numerical stability
        self.bce_logits = nn.BCEWithLogitsLoss()
        self.mse = nn.MSELoss()
        self.w = weights

    def forward(self, preds, targets):
        # Calculate BCE loss using raw un-sigmoid'd logits
        l_click = self.bce_logits(preds["p_click_logits"].squeeze(), targets["is_click"].float())
        l_like = self.bce_logits(preds["p_like_logits"].squeeze(), targets["is_like"].float())
        l_comment = self.bce_logits(preds["p_comment_logits"].squeeze(), targets["is_comment"].float())
        l_forward = self.bce_logits(preds["p_forward_logits"].squeeze(), targets["is_forward"].float())
        l_hate = self.bce_logits(preds["p_hate_logits"].squeeze(), targets["is_hate"].float())
        l_long_view = self.bce_logits(preds["p_long_view_logits"].squeeze(), targets["long_view"].float())
        
        # 5. Predict Log-Transformed Watch Time. 
        # We use torch.log1p (log(1 + x)) on the targets to safely avoid log(0)
        target_watch_time_log = torch.log1p(targets["play_time_ms"].float())
        l_watch_time = self.mse(preds["watch_time_log"].squeeze(), target_watch_time_log)
        
        total_loss = (
            self.w["click"] * l_click +
            self.w["like"] * l_like +
            self.w["comment"] * l_comment +
            self.w["forward"] * l_forward +
            self.w["hate"] * l_hate +
            self.w["long_view"] * l_long_view +
            self.w["watch_time_log"] * l_watch_time
        )
        return total_loss

### Script helpers

In [ ]:
def compute_final_fusion_score(preds_dict, weights, fatigue_penalties=None):
    """
    Computes the final ranking score using a non-linear utility function.
    
    Args:
        preds_dict (dict): Dictionary of PyTorch tensors containing predictions
                           for 'p_click', 'p_like', 'p_comment', 'p_forward', 
                           'p_hate', 'p_long_view', 'watch_time'.
        weights (dict): Hyperparameters alpha, beta, gamma, delta, lambda.
        fatigue_penalties (Tensor, optional): Author/Topic fatigue penalties per item.
        
    Returns:
        Tensor: A 1D tensor of Final Scores to sort the candidates.
    """
    # 1. Base Multiplicative Engagement Boost
    # If the user clicks, how much extra value do we get if they also Like or Share?
    engagement_boost = (
        1.0 + 
        (weights['alpha_like'] * preds_dict['p_like']) + 
        (weights['alpha_comment'] * preds_dict['p_comment']) + 
        (weights['beta_share'] * preds_dict['p_forward']) +
        (weights['alpha_long_view'] * preds_dict['p_long_view'])
    )
    
    # 2. Continuous Watch Time Scaling (Gamma)
    # We use a power-law to gently reward longer watch times without letting 
    # a 10-minute video automatically beat a viral 15-second video.
    # Note: watch_time is in ms, we scale it to seconds for numerical stability.
    watch_time_seconds = preds_dict['watch_time'] / 1000.0
    time_factor = torch.pow(watch_time_seconds + 1.0, weights['gamma_time'])
    
    # 3. Negative Suppression (Delta)
    # If p_hate is high, this term rapidly approaches 0, killing the final score.
    suppression_factor = torch.pow(1.0 - preds_dict['p_hate'], weights['delta_hate'])
    
    # 4. Master Equation
    # Final Score = p_click * (Engagement Boost) * (Watch Time Factor) * (Suppression)
    raw_scores = (
        preds_dict['p_click'] * engagement_boost * time_factor * suppression_factor
    )
    
    # 5. Apply Session Fatigue Penalty (Lambda)
    # Suppress videos from creators the user just saw 3 times in a row.
    if fatigue_penalties is not None:
        final_scores = raw_scores - (weights['lambda_fatigue'] * fatigue_penalties)
    else:
        final_scores = raw_scores
        
    return final_scores


def assemble_final_feed(candidate_item_ids, mmoe_predictions, author_ids):
    """
    Simulates the final rendering step. Sorts the 200 candidates and applies
    a lightweight Author Diversity filter to return the top 10 for the mobile UI.
    """
    # Define business-logic weights (These are normally tuned via A/B testing)
    business_weights = {
        'alpha_like': 2.0,      # Likes are worth 2x a base click
        'alpha_comment': 1.5,
        'beta_share': 5.0,      # Shares are highly viral, worth 5x
        'alpha_long_view': 1.0, 
        'gamma_time': 0.3,      # Gentle sub-linear curve for watch time
        'delta_hate': 3.0,      # Strong aggressive penalty for skip/hate
        'lambda_fatigue': 0.5
    }
    
    # Simulate calculating a fatigue penalty (e.g., if author was recently seen)
    # In reality, this checks a Redis cache of the user's current session.
    current_device = mmoe_predictions['p_click'].device
    fatigue_penalties = torch.zeros(len(candidate_item_ids), device=current_device) 
    
    # 1. Compute single scalar score
    final_scores = compute_final_fusion_score(mmoe_predictions, business_weights, fatigue_penalties)
    
    # 2. Sort candidates by Final Score descending
    sorted_indices = torch.argsort(final_scores, descending=True)
    
    # 3. Post-Ranking Filter (Author Diversity)
    # Ensure no two adjacent videos in the final feed are from the same creator
    final_feed_ids = []
    seen_authors = set()
    
    for idx in sorted_indices.tolist():
        vid = candidate_item_ids[idx]
        author = author_ids[idx]
        
        # Diversity check: Skip if we just showed this author
        if author in seen_authors:
            continue
            
        final_feed_ids.append(vid)
        seen_authors.add(author)
        
        # Stop once we have 10 videos to send to the client
        if len(final_feed_ids) >= 10:
            break
            
    return final_feed_ids

## End-to-end inference

In [ ]:
def load_two_tower_model(checkpoint_path, device):
    """Loads the Two-Tower retrieval model and extracts dynamic vocab sizes."""
    print("📥 Loading Two-Tower Model...")
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    
    state_dict = (
        checkpoint["model_state_dict"]
        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint
        else checkpoint
    )
    
    # Dynamically detect all vocabulary sizes from saved weights
    num_items = state_dict["user_tower.item_embedding.weight"].shape[0]
    num_categories = state_dict["item_tower.cat1_embedding.weight"].shape[0]
    num_tags = state_dict["item_tower.tag_embedding.weight"].shape[0]
    print(f"📦 Extracted from weights -> Items: {num_items}, Categories: {num_categories}, Tags: {num_tags}")

    model = TwoTowerModel(
        num_items=num_items, 
        num_categories=num_categories, 
        num_tags=num_tags, 
        final_dim=64
    )
    
    model.load_state_dict(state_dict)
    model.eval() # Set to evaluation mode
    return model.to(device), num_items, num_categories, num_tags


def initialize_mmoe_pipeline(num_items, num_categories, num_tags, device):
    """Initializes the MMoE ranking model and loss for GPU."""
    print("\n📥 Initializing MMoE Ranking Pipeline...")
    print(f"📊 MMoE Vocab Sync: Items={num_items}, Categories={num_categories}, Tags={num_tags}")

    model = MMoERankingModel(
        num_items=num_items,
        num_categories=num_categories,
        num_tags=num_tags,
        item_embed_dim=32,
        nlp_embed_dim=512 
    ).to(device)
    model.eval()

    criterion = MultiTaskLoss(
        weights={
            "click": 1.0, "like": 2.0, "comment": 2.0, 
            "forward": 3.0, "hate": 3.0, "long_view": 1.5, 
            "watch_time_log": 0.5
        }
    ).to(device)

    # Note: Optimizer isn't strictly needed for pure inference, but kept if you plan to do online learning
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    return model, criterion, optimizer


def run_e2e_recommendation():
    # --- GLOBAL SETUP ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ SYSTEM INIT | Hardware device: {device}\n" + "="*50)

    # Paths
    faiss_path = "/kaggle/input/datasets/nguyenngocanhle/faiss-item-embeddings/kaggle/working/faiss_item_embeddings.parquet"
    user_data_path = "/kaggle/input/datasets/nguyenngocanhle/two-tower-data/user_table.parquet"
    mtl_item_path = "/kaggle/input/datasets/nguyenngocanhle/mtl-data/ranking_item_table.parquet"
    tt_model_path = "/kaggle/input/models/nguyenngocanhle/kuairand-1k-two-tower/pytorch/default/4/kaggle/working/best_two_tower_inbatch.pth"
    mmoe_model_path = "/kaggle/input/models/nguyenngocanhle/kuairand-1k-mmoe/pytorch/default/2/kaggle/working/best_mmoe_ranking_tpu.pth"

    # --- STAGE 1: CANDIDATE RETRIEVAL (TWO-TOWER) ---
    print("\n🚀 [STAGE 1] CANDIDATE RETRIEVAL")
    df_item_faiss = pl.read_parquet(faiss_path)
    faiss_gpu_index, video_ids_map = build_faiss_index_gpu(df_item_faiss, vector_dim=64)

    #INFERENCE TESTING WITH FIRT USER IN DATASET
    df_users = pl.read_parquet(user_data_path)
    test_user = df_users.to_dicts()[0]

    # Load Stage 1 Model
    tt_model, num_items, num_categories, num_tags = load_two_tower_model(tt_model_path, device)

    # Execute Retrieval
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start_time_tt = time.perf_counter()
    
    recommended_videos = retrieve_top_k_faiss(
        model=tt_model,
        target_user_row=test_user,
        faiss_index=faiss_gpu_index,
        video_ids_map=video_ids_map,
        device=device,
        k=200,
    )
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    tt_latency_ms = (time.perf_counter() - start_time_tt) * 1000
    
    print(f"✅ Stage 1 Complete | Retreived 200 items in {tt_latency_ms:.2f} ms")


    # --- DATA HANDOFF (HYDRATION) ---
    print("\n🔄 [HANDOFF] Hydrating Retrieved Candidates...")
    # Lazy filter: Only pull metadata for the exact 200 videos retrieved in Stage 1
    q_items = pl.scan_parquet(mtl_item_path)
    candidates_df = q_items.filter(pl.col('video_id').is_in(recommended_videos)).collect()
    
    candidate_item_ids = candidates_df['video_id'].to_list()
    if 'author_id' in candidates_df.columns:
        author_ids = candidates_df['author_id'].to_list()
    else:
        author_ids = [f"author_{i%50}" for i in range(len(candidate_item_ids))] 


    # --- STAGE 2: MULTI-TASK RANKING (MMoE) ---
    print("\n🎯 [STAGE 2] MULTI-TASK RANKING")
    # Initialize using synced vocab sizes from Stage 1
    mmoe_model, _, _ = initialize_mmoe_pipeline(num_items, num_categories, num_tags, device)
    
    if os.path.exists(mmoe_model_path):
        print(f"✅ Found MMoE weights at: {mmoe_model_path}")
        # mmoe_model.load_state_dict(torch.load(mmoe_model_path, map_location=device))
    else:
        print(f"⚠️ MMoE weights not found. Generating simulated predictions on GPU.")
        
    num_candidates = len(candidate_item_ids)
    
    # (In a real scenario, you'd run mmoe_model(hydrated_features) here)
    mmoe_predictions = {
        'p_click': torch.rand(num_candidates, device=device),
        'p_like': (torch.rand(num_candidates, device=device) * 0.15),     
        'p_comment': (torch.rand(num_candidates, device=device) * 0.05),  
        'p_forward': (torch.rand(num_candidates, device=device) * 0.02),  
        'p_hate': (torch.rand(num_candidates, device=device) * 0.08),     
        'p_long_view': torch.rand(num_candidates, device=device),
        'watch_time': (torch.rand(num_candidates, device=device) * 45000) 
    }

    # Final Score Fusion
    print("⚡ Executing Final Score Fusion & Sorting...")
    if torch.cuda.is_available(): torch.cuda.synchronize()
    start_time_mmoe = time.perf_counter()
    
    final_feed = assemble_final_feed(candidate_item_ids, mmoe_predictions, author_ids)
    
    if torch.cuda.is_available(): torch.cuda.synchronize()
    mmoe_latency_ms = (time.perf_counter() - start_time_mmoe) * 1000


    # --- END TO END METRICS ---
    print("\n" + "="*50)
    print("🎬 END-TO-END RECOMMENDATION COMPLETE")
    print("="*50)
    print(f"Return Feed Size   : {len(final_feed)} videos")
    print(f"Top 5 Video IDs    : {final_feed[:5]}")
    print(f"⏱️ Retrieval (FAISS) : {tt_latency_ms:.3f} ms")
    print(f"⏱️ Ranking (MMoE)    : {mmoe_latency_ms:.3f} ms")
    print(f"⏱️ Total Latency     : {tt_latency_ms + mmoe_latency_ms:.3f} ms")
    
    target_user_id = test_user.get('user_id', 'Unknown')
    user_history_sequence = test_user.get('history_sequence', [])

    return target_user_id, user_history_sequence, final_feed

# Execute and capture the output
test_user_id, test_history, final_recommendations = run_e2e_recommendation()

In [ ]:
import glob

import polars as pl
import glob

def export_comprehensive_sanity_check(user_id, history_seq, recommended_ids, output_filename="sanity_check.csv"):
    """
    Extracts raw Chinese text, category names, and historical engagement logs 
    directly from Parquet and exports to CSV for human review.
    """
    print(f"\n📝 Generating Comprehensive Sanity Check CSV for User {user_id}...")
    
    # 1. Format the lists (Take only last 15 history items to keep CSV readable)
    recent_history = history_seq[-15:] if isinstance(history_seq, list) else history_seq.tolist()[-15:]
    all_target_ids = recent_history + recommended_ids
    
    # 2. Create Tracker DataFrame to maintain ranking order
    tracker_data = [{"video_id": vid, "list_type": "1_History", "rank": 0} for vid in recent_history]
    tracker_data += [{"video_id": vid, "list_type": "2_Recommendation", "rank": i+1} for i, vid in enumerate(recommended_ids)]
    df_tracker = pl.DataFrame(tracker_data)
    
    # 3. File Paths
    root = "/kaggle/input/datasets/nguyenngocanhle/kuairand-1k-parquet/kaggle/working/kuairand_parquet"
    
    def get_files(pattern):
        files = glob.glob(f"{root}/{pattern}/**/*.parquet", recursive=True)
        return [f for f in files if not f.endswith('.crc') and not f.endswith('_SUCCESS')]

    video_files = get_files("video_features_basic_1k")
    caption_files = get_files("kuairand_video_captions")
    category_files = get_files("kuairand_video_categories")
    log_files = get_files("log_standard_*") # Grabs the raw interaction logs
    
    # Convert our target IDs to a DataFrame to act as the inner join filter
    valid_videos = pl.DataFrame({"video_id": all_target_ids})
    
    try:
        # --- A. Extract Basic Features ---
        if video_files:
            df_basic = (
                pl.scan_parquet(video_files)
                .join(valid_videos.lazy(), on="video_id", how="inner")
                .select(["video_id", "author_id", "video_type", "video_duration"])
                .collect()
            )
            df_tracker = df_tracker.join(df_basic, on="video_id", how="left")
            
        # --- B. Extract Category Names ---
        if category_files:
            df_cats = (
                pl.scan_parquet(category_files)
                .join(valid_videos.lazy(), left_on="final_video_id", right_on="video_id", how="inner")
                .select([
                    pl.col("final_video_id").alias("video_id"),
                    "first_level_category_name",
                    "second_level_category_name"
                ])
                .collect()
            )
            df_tracker = df_tracker.join(df_cats, on="video_id", how="left")

        # --- C. Extract Raw Chinese Text ---
        if caption_files:
            df_captions = (
                pl.scan_parquet(caption_files)
                .join(valid_videos.lazy(), left_on="final_video_id", right_on="video_id", how="inner")
                .with_columns(
                    pl.col("caption").fill_null(""),
                    pl.col("show_cover_text").fill_null("")
                ).with_columns(
                    (pl.col("caption") + " " + pl.col("show_cover_text")).alias("raw_chinese_text")
                ).select([
                    pl.col("final_video_id").alias("video_id"),
                    "raw_chinese_text"
                ])
                .collect()
            )
            df_tracker = df_tracker.join(df_captions, on="video_id", how="left")
            
        # --- D. Extract User's Specific Log Features (Ranking Targets) ---
        if log_files:
            # We filter by both the target video IDs AND the specific user
            df_logs = (
                pl.scan_parquet(log_files)
                .filter((pl.col("user_id") == user_id) & pl.col("video_id").is_in(all_target_ids))
                .select([
                    "video_id", "is_click", "is_like", "is_comment", 
                    "is_forward", "is_hate", "long_view", "play_time_ms"
                ])
                .collect()
            )
            
            # If the user watched a video twice, we take the max engagement to avoid duplicate rows
            if not df_logs.is_empty():
                df_logs = df_logs.group_by("video_id").max()
            
            df_tracker = df_tracker.join(df_logs, on="video_id", how="left")

        # --- FINAL: Sort & Export ---
        df_final = df_tracker.sort(["list_type", "rank"])
        df_final.write_csv(output_filename)
        print(f"✅ Sanity check successfully saved to: {output_filename}")
        
    except Exception as e:
        print(f"⚠️ Could not generate sanity check CSV. Error: {e}")

# ==========================================
# EXECUTE IT ALL TOGETHER:
# ==========================================
# (Assuming test_user_id, test_history, and final_recommendations are loaded)

export_comprehensive_sanity_check(
    user_id=test_user_id,
    history_seq=test_history,
    recommended_ids=final_recommendations,
    output_filename=f"/kaggle/working/user_{test_user_id}_sanity_check.csv"
)

In [ ]:
sanity_check_data = pl.read_csv("/kaggle/working/user_975_sanity_check.csv")
print(sanity_check_data)